In [ ]:
from obspy import UTCDateTime, read as obspy_read
from obspy.clients.fdsn import Client
import folium
import pandas as pd
from IPython.display import display, HTML
import datetime

In [ ]:
# Get event data on birthday
def get_events(birthday):
    # convert to UTCDateTime
    bdayUTC = UTCDateTime(birthday.replace("/",","))

    # query USGS catalog
    client = Client("USGS")
    all_events_catalog = client.get_events(
        starttime=bdayUTC,
        endtime=bdayUTC + datetime.timedelta(days=1)
        )
    # convert catalog to a pandas dataframe
    df = pd.DataFrame([{
        'time': (event.origins[0]).time,
        'latitude': (event.origins[0]).latitude,
        'longitude': (event.origins[0]).longitude,
        'depth': (event.origins[0]).depth,
        'magnitude': (event.magnitudes[0]).mag if event.magnitudes else None
    } for event in all_events_catalog])
    
    # sort event by magnitude
    df = df.sort_values(by=['magnitude'], ascending=[False])

    return df

def map_it(birthday):

    # get events
    df = get_events(birthday)
    
    # get location of event with greatest magnitude
    latitude = df.loc[df.index[0], 'latitude']
    longitude = df.loc[df.index[0], 'longitude']
    
    # Create a map centered on a specific location (e.g., Alps)
    m = folium.Map(location=[latitude, longitude], 
                   zoom_start=2,
                   tiles='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
                   # tiles='Esri.WorldImagery',
                   name = 'Google Satellite',
                   attr='Google',
                   overlay = True,
                   control = True)
    
    folium.Marker(
        location=[latitude, longitude],
        popup="<b>It's your birthquake!</b>", # HTML can be used in popups
        tooltip="Click for info", # Tooltip appears on hover
        icon=folium.Icon(color='green', icon='info-sign') # Customize the marker's appearance
    ).add_to(m)
    
    # Display top 3 events by magnitude
    formatted_text = "<b>The biggest earthquakes on your birthday were:</b>"
    display(HTML(f"<p style='color:blue;'>{formatted_text}</p>"))
    display(df.head(3))
    
    # Display the map
    display(m)

In [ ]:

map_it(input("Enter your birthday: (yyyy/mm/dd)"))
